# Lab6.2_FPGA — Resize + Grayscale + Sobel în FPGA
### Experimente complete A / B / C — PYNQ-Z2, PYNQ 2.7.0, Vivado 2022.2

**Un singur notebook, executabil de la început până la sfârșit.** Nu necesită execuția unui notebook anterior și nu depinde de `lab6_common.py`.

**Arhitectură implementată în overlay-ul experimental:** `BGR / DDR → AXI DMA MM2S → Resize (PL) → Grayscale + Sobel (PL) → AXI DMA S2MM → DDR`. Verificarea hardware și rezultatele numerice trebuie obținute efectiv pe PYNQ-Z2; existența overlay-ului nu le certifică.

| Secțiune | Intrare | Rezultate |
|:--|:--|:--|
| **A1** | Imagine sintetică | Validare numerică OpenCV/FPGA și vizualizare |
| **A2** | 8 cadre sintetice deja în RAM | **200 de procesări**, timp și capacitate de procesare, fără captură |
| **B** | JPG / PNG de pe disc | Imagine ARM/FPGA/diferență și benchmark individual |
| **C1** | Video MP4/AVI sau cameră USB | Captură și verificare primului cadru |
| **C2** | Aceleași cadre video stocate în RAM | Procesare ARM vs FPGA fără blocajul camerei |
| **C3** | Citire video/cameră în timp real | Citire, pregătire, procesare, decizie, end-to-end, FPS |
| **C4** | Aceeași sursă video | Export opțional al rezultatului FPGA ca AVI |

> **Important:** camera USB poate limita FPS-ul întregii aplicații. De aceea se raportează **separat** accelerarea procesării (`ARM_ms / FPGA_ms`) și accelerarea buclei video (`FPS_FPGA / FPS_ARM`). Nu sunt aceeași metrică.

**Condiții:** fișierele `resize_sobel.bit` și `resize_sobel.hwh` din **același build** trebuie să se afle în `Lab6/overlay/` sau lângă notebook. După un timeout DMA, reporniți kernelul Jupyter. Nu rulați simultan alt notebook care reprogramează FPGA-ul.

## 0. Configurația experimentului

Pentru camera USB: `VIDEO_MODE = "camera"`; `VIDEO_FILE` este ignorat. Pentru fișier: `VIDEO_MODE = "file"` și încărcați `video.mp4` în directorul notebook-ului. Valoarea inițială este configurată pentru **camera USB**, conform experimentului curent.

Citirea din fișier și achiziția de la cameră se efectuează numai în secțiunea C. Benchmark-urile A2/C2 procesează cadrele **deja aflate în RAM**. Recomandăm să păstrați inițial rezoluțiile 640×480 la intrare și 320×240 la ieșire.

In [ ]:
from pathlib import Path
from time import perf_counter, sleep
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

VIDEO_MODE = "camera"        # "camera" sau "file"
VIDEO_FILE = Path("video.mp4")
CAMERA_INDEX = 0             # /dev/video0
CAMERA_FOURCC = "MJPG"      # format utilizat cu camera USB testată
INPUT_SIZE = (640, 480)     # (lățime, înălțime) la DMA
OUTPUT_SIZE = (320, 240)   # (lățime, înălțime) la ieșire

IMAGE_FILES = [Path("../Lab4/images/fruits.jpg")]
for extension in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"):
    IMAGE_FILES.extend(sorted(Path("images").glob(extension)))
# Descoperă automat imaginile adăugate în Lab6/images/
N_RAM = 200                # apeluri ARM și FPGA pe cadre din RAM
N_STORED = 8               # cadre păstrate în RAM, nu N_RAM cadre complete
N_VIDEO = 50               # cadre pe fiecare traseu pentru bucla video
CAMERA_WARMUP = 15         # cadre ignorate numai la pornirea camerei
EDGE_THRESHOLD = 80
SAVE_VIDEO = True
OUTPUT_VIDEO = Path("Lab6.2_FPGA_video.avi")

print("OpenCV:", cv2.__version__)
print("Sursa video:", VIDEO_MODE, "| intrare:", INPUT_SIZE, "| ieșire:", OUTPUT_SIZE)


## 1. Încărcarea overlay-ului și funcțiile comune

Identificăm blocurile din `overlay.ip_dict` și controlăm IP-urile prin `register_map` (fără adrese fizice hard-codate). Buffer-ele de intrare și ieșire sunt alocate în memorie contiguă, reutilizate când geometria rămâne neschimbată.

**Secvență de lansare:** armăm canalul S2MM → pornim Sobel → pornim Resize → lansăm MM2S. Verificăm cu timeout de 5 secunde fiecare canal DMA. `0x01` la registrul de control înseamnă `ap_start` *one-shot*, fără auto-restart.

In [ ]:
def locate_overlay():
    here = Path.cwd()
    paths = [here / "overlay" / "resize_sobel.bit", here / "resize_sobel.bit",
             here.parent / "Lab6" / "overlay" / "resize_sobel.bit"]
    for bit in paths:
        if bit.is_file() and bit.with_suffix(".hwh").is_file():
            return bit.resolve()
    raise FileNotFoundError(
        "Lipsesc resize_sobel.bit și resize_sobel.hwh. "
        "Copiați ambele fișiere în Lab6/overlay/.")


def wait_dma(channel, name, timeout_s=5.0):
    start = perf_counter()
    while not channel.idle:
        if getattr(channel, "error", False):
            raise RuntimeError("DMA {} raportează eroare; reporniți kernelul.".format(name))
        if perf_counter() - start >= timeout_s:
            raise TimeoutError("Timeout DMA {} după {} s. Reporniți kernelul Jupyter."
                               .format(name, timeout_s))
        sleep(0.001)
    channel.wait()


class FusedResizeSobel:
    def __init__(self, overlay_obj):
        for ip in ("axi_dma_0", "resize_accel_0", "sobel_accel_0"):
            if ip not in overlay_obj.ip_dict:
                raise RuntimeError("IP absent din overlay: " + ip)
        self.dma = overlay_obj.axi_dma_0
        self.resize = overlay_obj.resize_accel_0
        self.sobel = overlay_obj.sobel_accel_0
        self.geom = None
        self.src = None
        self.dst = None
        self.last_breakdown = {}

    def close(self):
        for field in ("src", "dst"):
            buf = getattr(self, field)
            if buf is not None:
                buf.freebuffer()
                setattr(self, field, None)
        self.geom = None

    def run(self, frame, size=OUTPUT_SIZE):
        if not isinstance(frame, np.ndarray) or frame.dtype != np.uint8 or \
                frame.ndim != 3 or frame.shape[2] != 3:
            raise ValueError("Intrarea trebuie să fie imagine BGR uint8 cu 3 canale.")
        height, width = frame.shape[:2]
        out_width, out_height = map(int, size)
        if not (1 <= width <= 3840 and 1 <= height <= 2160 and
                1 <= out_width <= 1920 and 1 <= out_height <= 1080):
            raise ValueError("Dimensiuni în afara limitelor configurate pentru IP.")
        if (height * width * 3) % 4 or (out_height * out_width) % 4:
            raise ValueError("Lungimile DMA trebuie să fie multipli de 4 octeți (DRE=0).")

        geometry = (height, width, out_height, out_width)
        if geometry != self.geom:
            self.close()
            self.src = allocate(shape=(height, width, 3), dtype=np.uint8)
            self.dst = allocate(shape=(out_height, out_width), dtype=np.uint8)
            self.geom = geometry

        t0 = perf_counter()
        np.copyto(self.src, np.ascontiguousarray(frame))
        self.src.flush()
        t1 = perf_counter()
        self.dst.fill(0)
        self.dst.flush()
        r = self.resize.register_map
        r.src_rows, r.src_cols = height, width
        r.dst_rows, r.dst_cols = out_height, out_width
        s = self.sobel.register_map
        s.rows, s.cols = out_height, out_width
        t2 = perf_counter()

        self.dma.recvchannel.transfer(self.dst)
        self.sobel.write(0x00, 0x01)
        self.resize.write(0x00, 0x01)
        self.dma.sendchannel.transfer(self.src)
        t3 = perf_counter()
        wait_dma(self.dma.sendchannel, "MM2S")
        wait_dma(self.dma.recvchannel, "S2MM")
        t4 = perf_counter()
        self.dst.invalidate()
        output = np.array(self.dst, copy=True)
        t5 = perf_counter()

        self.last_breakdown = {
            "copy_flush_ms": (t1-t0)*1000,
            "config_ms": (t2-t1)*1000,
            "launch_ms": (t3-t2)*1000,
            "wait_dma_ms": (t4-t3)*1000,
            "readback_ms": (t5-t4)*1000,
            "total_ms": (t5-t0)*1000
        }
        return output


def process_arm(frame, size=OUTPUT_SIZE):
    small = cv2.resize(frame, size, interpolation=cv2.INTER_LINEAR)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3,
                   borderType=cv2.BORDER_REFLECT_101)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3,
                   borderType=cv2.BORDER_REFLECT_101)
    return cv2.convertScaleAbs(cv2.magnitude(gx, gy))


def compare_outputs(ref, actual):
    if ref.shape != actual.shape:
        raise ValueError("Dimensiuni diferite ARM/FPGA: {} / {}".format(ref.shape, actual.shape))
    diff = np.abs(ref.astype(np.int16) - actual.astype(np.int16))
    return {"mae": float(diff.mean()), "max": int(diff.max()),
            "different": float(np.mean(diff != 0)), "nonzero": int(np.count_nonzero(actual))}


def show_comparison(frame, ref, actual, title):
    difference = np.abs(ref.astype(np.int16) - actual.astype(np.int16))
    images = [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), ref, actual, difference]
    labels = ["Intrare BGR", "Referință ARM", "FPGA Resize+Sobel", "Diferență absolută"]
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
    for i, ax in enumerate(axes):
        ax.imshow(images[i], **({} if i == 0 else {"cmap": "gray", "vmin": 0,
                                                   "vmax": 255 if i < 3 else max(1, int(difference.max()))}))
        ax.set_title(labels[i])
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def normalize(frame, size=INPUT_SIZE):
    if frame is None or frame.ndim != 3 or frame.shape[2] != 3 or frame.dtype != np.uint8:
        raise ValueError("Intrare video invalidă (se așteaptă BGR uint8).")
    if (frame.shape[1], frame.shape[0]) != tuple(size):
        frame = cv2.resize(frame, size, interpolation=cv2.INTER_LINEAR)
    return np.ascontiguousarray(frame)


BITFILE = locate_overlay()
ol = Overlay(str(BITFILE))
hw = FusedResizeSobel(ol)
print("Overlay:", BITFILE)
for ip in ("axi_dma_0", "resize_accel_0", "sobel_accel_0"):
    print(ip, "→", ol.ip_dict[ip]["type"])
print("Resize:", hw.resize.register_map)
print("Sobel:", hw.sobel.register_map)


## A1. Validare hardware pe imagine sintetică (64×48 → 32×24)

Testul separă erorile de hardware de problemele capturii video. Raportăm MAE și diferența maximă; nu impunem identitate exactă deoarece implementarea HLS poate avea rotunjiri diferite. **Dacă rezultatul FPGA este integral zero pentru acest test, nu continuați benchmark-urile.**

In [ ]:
ys, xs = np.indices((48, 64))
test = np.empty((48, 64, 3), dtype=np.uint8)
test[:, :, 0] = (4*xs + 2*ys) % 256
test[:, :, 1] = (7*xs + 3*ys) % 256
test[:, :, 2] = ((xs//8) ^ (ys//6)) * 40

ref_a1 = process_arm(test, (32, 24))
out_a1 = hw.run(test, (32, 24))
qa1 = compare_outputs(ref_a1, out_a1)
print("A1 | formă FPGA:", out_a1.shape, "| dtype:", out_a1.dtype)
print("A1 | MAE={mae:.3f} | max={max} | fracție diferită={different:.2%} | nenuli={nonzero}".format(**qa1))
show_comparison(test, ref_a1, out_a1, "A1 — Validare sintetică")
if qa1["nonzero"] == 0:
    raise RuntimeError("Ieșire FPGA integral zero. Opriți testele; verificați overlay și DMA.")
print("A1: ieșire FPGA nenulă; examinați și diferențele numerice înainte de a continua.")


## A2. Benchmark de procesare fără cameră și fără disc

Generăm 8 cadre BGR diferite **în RAM** și efectuăm 200 de apeluri pentru fiecare cale, pe **aceleași cadre**, în ordine intercalată ARM/FPGA. După 5 iterații de încălzire, măsurăm timpul apelului complet de procesare. Pentru FPGA acesta include copierea în buffer, configurarea registrelor, lansarea IP-urilor, DMA, așteptarea și preluarea rezultatului; **nu doar execuția logicii PL**.

FPS-ul afișat mai jos este **capacitatea teoretică observată de procesare serială**, `1000 / durata medie [ms]`, fără cameră și fără afișare, **nu** FPS-ul end-to-end al camerei.

In [ ]:
yy, xx = np.indices((INPUT_SIZE[1], INPUT_SIZE[0]))
base = np.empty((INPUT_SIZE[1], INPUT_SIZE[0], 3), dtype=np.uint8)
base[:, :, 0] = (2*xx + yy) % 256
base[:, :, 1] = (5*xx + 3*yy) % 256
base[:, :, 2] = (((xx//24) ^ (yy//20)) * 29) % 256
ram_frames = [np.ascontiguousarray(np.roll(base, shift=i*7, axis=1))
              for i in range(N_STORED)]


def benchmark_ram(frames, n=N_RAM, size=OUTPUT_SIZE):
    if not frames:
        raise ValueError("Nu există cadre în RAM.")
    for i in range(5):
        f = frames[i % len(frames)]
        process_arm(f, size)
        hw.run(f, size)
    cpu_ms, fpga_ms = [], []
    for i in range(n):
        f = frames[i % len(frames)]
        t0 = perf_counter()
        process_arm(f, size)
        cpu_ms.append(1000*(perf_counter()-t0))
        t0 = perf_counter()
        hw.run(f, size)
        fpga_ms.append(1000*(perf_counter()-t0))
    return np.array(cpu_ms), np.array(fpga_ms)


def print_ram_benchmark(title, cpu_ms, fpga_ms):
    print("\n=== {} ===".format(title))
    print("{:18s} {:>12s} {:>12s}".format("Indicator", "ARM", "FPGA"))
    print("{:18s} {:12.3f} {:12.3f}".format("Medie [ms]", cpu_ms.mean(), fpga_ms.mean()))
    print("{:18s} {:12.3f} {:12.3f}".format("Mediană [ms]", np.median(cpu_ms), np.median(fpga_ms)))
    print("{:18s} {:12.3f} {:12.3f}".format("P95 [ms]", np.percentile(cpu_ms,95), np.percentile(fpga_ms,95)))
    print("{:18s} {:12.2f} {:12.2f}".format("FPS procesare", 1000/cpu_ms.mean(), 1000/fpga_ms.mean()))
    print("Accelerare procesare: {:.2f}x".format(cpu_ms.mean()/fpga_ms.mean()))

cpu_a2, fpga_a2 = benchmark_ram(ram_frames)
print_ram_benchmark("A2 — {} cadre RAM".format(N_RAM), cpu_a2, fpga_a2)
print("Ultimul apel FPGA — etape driver [ms]:", hw.last_breakdown)


## B. Procesarea imaginilor JPG / PNG

Implicit încearcă `../Lab4/images/fruits.jpg`. Pentru imagini proprii, încărcați fișiere în `Lab6/images/` și vor fi descoperite automat; puteți specifica și alte căi în `IMAGE_FILES`.

Normalizarea la 640×480 și citirea de pe disc sunt **în afara** benchmark-ului B; datele sunt identice pentru ARM și FPGA. Dacă fișierul implicit lipsește, secțiunea B este omisă explicit, iar experimentele cu cameră din C pot continua.

In [ ]:
images_found = 0
results_b = []
for img_path in IMAGE_FILES:
    img_path = Path(img_path)
    if not img_path.is_file():
        print("B — fișier absent, omis:", img_path)
        continue
    original = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if original is None:
        print("B — fișier necitibil, omis:", img_path)
        continue
    frame = normalize(original)
    arm = process_arm(frame)
    fpga = hw.run(frame)
    quality = compare_outputs(arm, fpga)
    images_found += 1
    print("\nB:", img_path, "| intrare:", original.shape, "→", frame.shape)
    print("MAE={mae:.3f} | max={max} | fracție diferită={different:.2%} | nenuli FPGA={nonzero}".format(**quality))
    show_comparison(frame, arm, fpga, "B — " + img_path.name)
    if quality["nonzero"] == 0 and int(np.count_nonzero(arm)) > 0:
        raise RuntimeError("Rezultat FPGA invalid, integral zero, pentru " + str(img_path))
    arm_t, fpga_t = benchmark_ram([frame], n=20)
    print_ram_benchmark("B — " + img_path.name, arm_t, fpga_t)
    results_b.append({"name": img_path.name, "quality": quality,
                      "arm_ms": float(arm_t.mean()), "fpga_ms": float(fpga_t.mean())})

if images_found == 0:
    print("B nu a fost executată: adăugați un JPG/PNG și reluați secțiunea.")


## C1. Camera USB / fișier video — inițializare și controlul cadrului

Camera este deschisă prin Video4Linux2 (`CAP_V4L2`) și solicită `MJPG`, 640×480. Pentru o cameră USB, ignorăm **15 cadre** de la pornire (încălzire/autoexpunere); pentru fișier, începem din primul cadru. Nu este garantat că toate camerele acceptă formatul solicitat.

**Dacă imaginea este neagră**, verificați mai întâi **obturatorul mecanic** și iluminarea. Un MAE de 0 între două imagini negre nu validează acceleratorul.

In [ ]:
def open_source():
    if VIDEO_MODE == "camera":
        cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_V4L2)
        if not cap.isOpened():
            raise RuntimeError("Camera /dev/video{} nu poate fi deschisă.".format(CAMERA_INDEX))
        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*CAMERA_FOURCC))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, INPUT_SIZE[0])
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, INPUT_SIZE[1])
        for _ in range(CAMERA_WARMUP):
            ok, _ = cap.read()
            if not ok:
                cap.release()
                raise RuntimeError("Camera nu furnizează cadre la încălzire.")
    elif VIDEO_MODE == "file":
        if not VIDEO_FILE.is_file():
            raise FileNotFoundError("Fișier video absent: " + str(VIDEO_FILE))
        cap = cv2.VideoCapture(str(VIDEO_FILE))
        if not cap.isOpened():
            raise RuntimeError("Nu poate fi deschis fișierul video: " + str(VIDEO_FILE))
    else:
        raise ValueError("VIDEO_MODE trebuie să fie 'camera' sau 'file'.")
    return cap


cap = open_source()
try:
    actual_fps = cap.get(cv2.CAP_PROP_FPS)
    actual_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    current_fmt = ''.join(chr((fourcc >> (8*i)) & 255) for i in range(4))
    ok, first = cap.read()
finally:
    cap.release()
if not ok:
    raise RuntimeError("Nu a fost citit niciun cadru video.")
first_frame = normalize(first)
print("Cameră/sursă:", VIDEO_MODE, "| format:", current_fmt,
      "| dimensiune:", (actual_width, actual_height), "| FPS raportat:", actual_fps)
print("Cadru: min={}, max={}, medie={:.2f}".format(
    int(first.min()), int(first.max()), float(first.mean())))
if first.max() <= 15 and first.mean() < 3:
    raise RuntimeError("Cadru aproape negru. Verificați obturatorul mecanic al camerei!")

ref_c1 = process_arm(first_frame)
fpga_c1 = hw.run(first_frame)
qc1 = compare_outputs(ref_c1, fpga_c1)
print("C1 | MAE={mae:.3f} | max={max} | fracție diferită={different:.2%} | FPGA nenuli={nonzero}".format(**qc1))
show_comparison(first_frame, ref_c1, fpga_c1, "C1 — primul cadru video")
if qc1["nonzero"] == 0 and int(np.count_nonzero(ref_c1)) > 0:
    raise RuntimeError("C1: FPGA produce numai zero, deși referința ARM nu este zero.")


## C2. Benchmark al cadrelor video **în RAM** — fără așteptarea camerei

Capturăm o singură dată maximum 8 cadre video, după care **închidem camera**. Apoi procesăm aceleași cadre de 200 de ori prin ARM și FPGA, în ordine intercalată. Este cel mai relevant test pentru a separa capacitatea de procesare de blocajul de captură video.

Rezultatele sunt comparabile intern deoarece se folosesc aceiași pixeli. Captura inițială **nu** intră în cronometrare.

In [ ]:
cap = open_source()
video_frames = []
try:
    for _ in range(N_STORED):
        ok, captured = cap.read()
        if not ok:
            break
        video_frames.append(normalize(captured))
finally:
    cap.release()
if not video_frames:
    raise RuntimeError("Nu s-au capturat cadre pentru testul C2.")
print("C2 | Cadre memorate:", len(video_frames),
      "| memorie aproximativă: {:.1f} MiB".format(
          sum(f.nbytes for f in video_frames) / 1048576.0))
cpu_c2, fpga_c2 = benchmark_ram(video_frames, n=N_RAM)
print_ram_benchmark("C2 — {} procesări, cadre video în RAM".format(N_RAM), cpu_c2, fpga_c2)


## C3. Benchmark end-to-end cu achiziție video efectivă

Măsurăm separat:
1. `Citire` — `cap.read()` (așteptare cadru și/sau decodare);
2. `Prep` — redimensionare la 640×480 și conversia la buffer contiguu;
3. `Proces` — Resize + Grayscale + Sobel (ARM sau întregul pipeline în FPGA);
4. `Decizie` — fracția de pixeli care depășesc pragul `EDGE_THRESHOLD`;
5. `E2E` și FPS — procesare serială pentru bucla completă, **fără afișare sau export**.

Traseele ARM și FPGA redeschid pe rând sursa. Pentru **cameră** nu sunt aceleași cadre, iar timpul `read()` poate crește tocmai fiindcă FPGA termină mai repede și așteaptă cadrul următor. Pentru **fișier** se reia aceeași secvență de la început, fără încălzire. Raportul FPS nu dovedește singur accelerarea logicii.

In [ ]:
def benchmark_stream(label, processor, n=N_VIDEO):
    cap = open_source()
    samples = []
    start = perf_counter()
    try:
        for _ in range(n):
            t0 = perf_counter()
            ok, frame = cap.read()
            t1 = perf_counter()
            if not ok:
                break
            frame = normalize(frame)
            t2 = perf_counter()
            edges = processor(frame)
            t3 = perf_counter()
            density = float(np.mean(edges >= EDGE_THRESHOLD))
            t4 = perf_counter()
            samples.append([1000*(t1-t0), 1000*(t2-t1), 1000*(t3-t2),
                            1000*(t4-t3), 1000*(t4-t0), density])
    finally:
        cap.release()
    wall_s = perf_counter() - start
    if not samples:
        raise RuntimeError("Nu s-a procesat niciun cadru pentru: " + label)
    data = np.asarray(samples, dtype=np.float64)
    return {"name": label, "count": len(samples),
            "read": float(data[:,0].mean()), "prep": float(data[:,1].mean()),
            "proc": float(data[:,2].mean()), "decision": float(data[:,3].mean()),
            "e2e": float(data[:,4].mean()), "p95": float(np.percentile(data[:,4],95)),
            "fps": float(len(samples)/wall_s), "edge_density": float(data[:,5].mean())}


arm_stream = benchmark_stream("ARM", lambda f: process_arm(f))
fpga_stream = benchmark_stream("FPGA", lambda f: hw.run(f))

print("\n=== C3 — Video end-to-end: {} ===".format(VIDEO_MODE))
print("{:6s} {:>6s} {:>10s} {:>9s} {:>10s} {:>10s} {:>10s} {:>8s}".format(
    "Cale", "Cadre", "Citire ms", "Prep ms", "Proces ms", "Decizie ms", "E2E ms", "FPS"))
for r in [arm_stream, fpga_stream]:
    print("{name:6s} {count:6d} {read:10.2f} {prep:9.2f} {proc:10.2f} "
          "{decision:10.2f} {e2e:10.2f} {fps:8.2f}".format(**r))
    print("  p95 end-to-end={:.2f} ms | densitate muchii={:.2%}".format(
        r["p95"], r["edge_density"]))
print("Accelerare procesare C3: {:.2f}x".format(arm_stream["proc"]/fpga_stream["proc"]))
print("Raport FPS end-to-end C3: {:.2f}x".format(fpga_stream["fps"]/arm_stream["fps"]))
print("Diferență timp citire (FPGA-ARM): {:+.2f} ms".format(
    fpga_stream["read"]-arm_stream["read"]))
if VIDEO_MODE == "camera":
    print("Interpretare: FPS-ul camerei poate plafona bucla, chiar când FPGA este mai rapid la procesare.")


## C4. Exportul videoclipului procesat de FPGA — opțional

Salvăm cadrele cu muchii Sobel în `Lab6.2_FPGA_video.avi` (MJPEG) în directorul notebook-ului, **în afara benchmark-ului**. Codec-ul AVI/MJPEG depinde de configurația OpenCV; dacă nu este disponibil, măsurătorile C2/C3 rămân valide. Dacă `SAVE_VIDEO = False`, această etapă este omisă.

In [ ]:
if SAVE_VIDEO:
    cap = open_source()
    writer = None
    written = 0
    try:
        nominal_fps = cap.get(cv2.CAP_PROP_FPS)
        out_fps = (float(fpga_stream["fps"]) if VIDEO_MODE == "camera"
                   else float(nominal_fps) if np.isfinite(nominal_fps)
                   and 1 <= nominal_fps <= 60 else 15.0)
        writer = cv2.VideoWriter(str(OUTPUT_VIDEO),
                                 cv2.VideoWriter_fourcc(*"MJPG"),
                                 out_fps, OUTPUT_SIZE, True)
        if not writer.isOpened():
            print("C4 — encoder MJPEG indisponibil; exportul a fost omis.")
        else:
            for _ in range(N_VIDEO):
                ok, frame = cap.read()
                if not ok:
                    break
                edges = hw.run(normalize(frame))
                writer.write(cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR))
                written += 1
            print("C4 — videoclip:", OUTPUT_VIDEO.resolve(), "| cadre:", written)
    finally:
        cap.release()
        if writer is not None:
            writer.release()
else:
    print("C4 — export dezactivat din configurație.")


## 2. Sinteză și interpretare

Raportul trebuie să distingă între **timpul apelului de procesare** și **timpul end-to-end**. Un factor de accelerare de 2,45× pentru procesare și un factor de ~1× la FPS-ul camerei sunt compatibile: bucla sincronă rămâne limitată de disponibilitatea cadrelor.

**Întrebări pentru studenți:**
1. Cu cât scade timpul de procesare când folosim pipeline-ul complet FPGA?
2. În C2 (cadre în RAM), ce FPS de procesare rezultă pentru ARM și FPGA?
3. În C3, cât din timpul total ocupă `cap.read()` și de ce poate fi mai mare pentru FPGA?
4. Ce diferență există între FPS-ul camerei, FPS-ul de procesare și latența end-to-end?
5. În ce condiții ar ajuta achiziția asincronă, buffering-ul sau execuția pe un flux hardware dedicat?

**Limitare:** benchmark-ul este pe o buclă Python sincronă. Rezultatele nu măsoară throughput-ul maxim intern al acceleratorului PL și nici utilizarea exactă a DDR. Nu extrapolați din timpii observați la alte rezoluții, interfețe sau platforme.

In [ ]:
print("\n=== REZUMAT LAB6.2 FPGA ===")
print("A2: accelerare procesare (sintetic în RAM): {:.2f}x".format(
    cpu_a2.mean()/fpga_a2.mean()))
print("C2: accelerare procesare (cadre video în RAM): {:.2f}x".format(
    cpu_c2.mean()/fpga_c2.mean()))
print("C3: accelerare procesare (sursă video): {:.2f}x".format(
    arm_stream["proc"]/fpga_stream["proc"]))
print("C3: raport FPS end-to-end: {:.2f}x".format(
    fpga_stream["fps"]/arm_stream["fps"]))
print("Metodologie: procesare pe cadre în RAM vs buclă video cu citire blocantă.")
hw.close()  # eliberează buffer-ele; obiectul poate realoca ulterior la nevoie
print("Buffer-ele DMA au fost eliberate. Laborator încheiat.")


## 3. Grafice comparative și concluzii experimentale

Această secțiune utilizează **exclusiv rezultatele calculate în celulele anterioare**. Nu sunt introduse valori de benchmark manual.

- **Procesare:** timpii apelurilor OpenCV/FPGA (copieri, DMA și acceleratoare incluse), fără achiziția camerei.
- **End-to-end C3:** citire blocantă, pregătire, procesare și decizie. Nu include afișare sau înregistrare video.
- **FPS C3:** numărul cadrelor procesate pe durata benchmark-ului. În implementarea curentă, timpul cronometrat include și `cap.release()` (dar nu deschiderea camerei).

**Limite metodologice:** ARM și FPGA citesc camera în sesiuni succesive și pot primi cadre diferite; măsurătorile trebuie repetate. Pentru video din fișier, performanța depinde de codec și de decodare. FPS-ul nominal al camerei nu reprezintă FPS-ul aplicației.


In [ ]:
# Graficele sunt generate din măsurătorile curente, fără valori hard-codate.
series = [("A2 — sintetic, RAM", float(cpu_a2.mean()), float(fpga_a2.mean()))]
for item in globals().get("results_b", []):
    series.append(("B — " + item["name"], item["arm_ms"], item["fpga_ms"]))
series.extend([
    ("C2 — video, RAM", float(cpu_c2.mean()), float(fpga_c2.mean())),
    ("C3 — video, procesare", arm_stream["proc"], fpga_stream["proc"])
])
names = [r[0] for r in series]
arm_ms = np.array([r[1] for r in series], dtype=float)
pl_ms = np.array([r[2] for r in series], dtype=float)
positions = np.arange(len(names))
fig, ax = plt.subplots(figsize=(11, max(4, len(names)*0.75+1)))
bars_a = ax.barh(positions-0.18, arm_ms, height=0.35, label="ARM / OpenCV")
bars_p = ax.barh(positions+0.18, pl_ms, height=0.35, label="FPGA / Resize + Sobel")
ax.set_yticks(positions)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel("Durată medie procesare [ms/cadru] — mai mic este mai bine")
ax.set_title("Lab6.2 — procesare ARM vs. FPGA")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
ax.set_xlim(0, max(arm_ms.max(), pl_ms.max())*1.2)
ax.legend()
for bar in list(bars_a)+list(bars_p):
    ax.text(bar.get_width()+0.15, bar.get_y()+bar.get_height()/2,
            "{:.2f}".format(bar.get_width()), va="center", fontsize=9)
plt.tight_layout()
plt.show()

# Distribuția duratelor în bucla video (C3).
fig, ax = plt.subplots(figsize=(8, 4.5))
bottom = np.zeros(2, dtype=float)
for label, key in [("Citire", "read"), ("Pregătire", "prep"),
                   ("Procesare", "proc"), ("Decizie", "decision")]:
    values = np.array([arm_stream[key], fpga_stream[key]], dtype=float)
    ax.bar(["ARM", "FPGA"], values, bottom=bottom, label=label)
    bottom += values
ax.set_ylabel("Latență medie end-to-end [ms/cadru]")
ax.set_title("Lab6.2 C3 — componentele latenței")
ax.set_ylim(0, bottom.max()*1.2)
ax.legend()
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
for i, val in enumerate(bottom):
    ax.text(i, val+1, "{:.2f} ms".format(val), ha="center")
plt.tight_layout()
plt.show()

# FPS măsurat și percentila P95 a latenței end-to-end.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title, ylabel in [
    (axes[0], [arm_stream["fps"], fpga_stream["fps"]],
     "C3 — FPS efectiv", "FPS"),
    (axes[1], [arm_stream["p95"], fpga_stream["p95"]],
     "C3 — latența P95", "ms/cadru")
]:
    ax.bar(["ARM", "FPGA"], values)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(values)*1.2)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    for i, val in enumerate(values):
        ax.text(i, val+max(values)*0.025, "{:.2f}".format(val), ha="center")
plt.tight_layout()
plt.show()


In [ ]:
# Concluziile folosesc numai datele acestei execuții.
speedup_ram = float(cpu_c2.mean()/fpga_c2.mean())
speedup_video = float(arm_stream["proc"]/fpga_stream["proc"])
reduction_e2e = 100*(1-fpga_stream["e2e"]/arm_stream["e2e"])
fps_gain = 100*(fpga_stream["fps"]/arm_stream["fps"]-1)
print("\n=== CONCLUZII Lab6.2_FPGA ===")
print("1. Accelerare procesare C2, pe cadre video în RAM: {:.2f}x.".format(speedup_ram))
print("2. Accelerare procesare C3, în bucla video: {:.2f}x.".format(speedup_video))
print("3. Reducerea latenței medii end-to-end C3: {:.1f}%.".format(reduction_e2e))
print("4. Creșterea FPS observat în C3: {:.1f}%.".format(fps_gain))
print("5. Citirea camerei — ARM: {:.2f} ms; FPGA: {:.2f} ms.".format(
    arm_stream["read"], fpga_stream["read"]))
print("6. Validare A1: MAE {:.3f}, maxdiff {}.".format(qa1["mae"], qa1["max"]))
print("7. Validare C1: MAE {:.3f}, maxdiff {}.".format(qc1["mae"], qc1["max"]))
print("8. FPS end-to-end nu este egal cu FPS de procesare: citirea video și decizia adaugă latență.")
print("9. Comparația ARM/FPGA pe camera live nu folosește exact aceleași cadre.")
print("10. Repetarea testului este necesară pentru statistici robuste.")
